<a href="https://colab.research.google.com/github/SnehAl2o7/DDRS-Deepshiva/blob/main/Deepshiva_DDRS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
# Install transformers branch for Ministral
!pip install git+https://github.com/huggingface/transformers.git@bf3f0ae70d0e902efab4b8517fce88f6697636ce
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is

In [ ]:
import torch
from unsloth import FastLanguageModel


max_seq_length = 4096
dtype = None
load_in_4bit = True # Enables 4-bit quantization (QLoRA)

# Load the Mistral 7B model and its tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mistralai/Mistral-7B-Instruct-v0.2", # Use instruct model for better performance
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Model loaded and quantized. dtype: {model.dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.0.0.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Model loaded and quantized. dtype: torch.float16


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# Check the number of trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of Trainable Parameters: {trainable_params / 1e6:.2f} Million")
print(f"Total Model Parameters: {total_params / 1e9:.2f} Billion")

# Define the local directory name where the model will be saved
SAVE_DIR = "Mistral_7B_parameter_model_Quantasized"

# 1. Save the model adapters (The LoRA weights)
model.save_pretrained(SAVE_DIR, save_adapter=True)

# 2. Save the tokenizer (Crucial for the chat format)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\n✅ Model and Tokenizer saved to: ./{SAVE_DIR}/")

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Number of Trainable Parameters: 41.94 Million
Total Model Parameters: 3.79 Billion


Unsloth: Restored added_tokens_decoder metadata in Mistral_7B_parameter_model_Quantasized/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in Mistral_7B_parameter_model_Quantasized.



✅ Model and Tokenizer saved to: ./Mistral_7B_parameter_model_Quantasized/


# Task
Test the loaded model with a basic conversation prompt to evaluate its response characteristics.

## Test Model with Basic Conversation

### Subtask:
Formulate a simple conversation prompt and use the loaded `model` and `tokenizer` to generate a response to understand its output characteristics.


In [ ]:
# messages = [
#     {"role": "user", "content": "What are the benefit's of tourism and give feedback on tourism in uttarakhand"}
# ]

# # 1. Create a basic conversation prompt string (already done in messages format)
# # 2. Use the tokenizer.apply_chat_template() method
# input_ids = tokenizer.apply_chat_template(messages,
#                                           tokenize=True,
#                                           add_generation_prompt=True,
#                                           return_tensors="pt")

# # 3. Move the generated token IDs to the GPU
# input_ids = input_ids.to("cuda")

# print("Generating response...")

# # 4. Generate a response using the model.generate() method
# outputs = model.generate(input_ids, max_new_tokens=256, use_cache=True)

# # 5. Decode the generated token IDs back into readable text
# response = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

# # 6. Print the generated response
# print(response)
# print("Model response generated.")

## Using the Fast API, uvicorn and ngrok to deploy the model locally.

the code is given below

In [ ]:
! pip install fastapi uvicorn nest-asyncio pyngrok -qq

In [ ]:
import os
from google.colab import userdata
from pyngrok import ngrok

# --- Configuration ---
NGROK_TOKEN_NAME = 'NGROK_AUTH_TOKEN'
LOCAL_PORT = 8000 # The port used by Uvicorn

# 1. Attempt to retrieve the secret value
try:
    auth_token = userdata.get(NGROK_TOKEN_NAME)

    if auth_token:
        # 2. If the token is found, set it for pyngrok
        ngrok.set_auth_token(auth_token)
        print(f"✅ ngrok authentication token set from Colab Secret: {NGROK_TOKEN_NAME}.")
    else:
        # This branch handles cases where the secret exists but the value is empty
        raise ValueError("The Colab Secret NGROK_AUTH_TOKEN was found, but its value is empty.")

except Exception as e:
    # This block catches errors like the secret not existing or other retrieval issues
    print(f"❌ ERROR: Failed to retrieve secret '{NGROK_TOKEN_NAME}'.")
    print("Please ensure you have set the secret in the 🔑 Secrets sidebar.")
    # Stop execution if the critical token is missing
    raise

✅ ngrok authentication token set from Colab Secret: NGROK_AUTH_TOKEN.


In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from contextlib import asynccontextmanager
import torch
from unsloth import FastLanguageModel
import os
import uvicorn
import nest_asyncio

# --- Configuration ---
MODEL_PATH = "/content/Mistral_7B_parameter_model_Quantasized"

# --- Global Variables for Model ---
model = None
tokenizer = None
max_seq_length = 4096

# --- Pydantic Model for API Request ---
class ChatRequest(BaseModel):
    """Defines the structure for the incoming chat request from the frontend."""
    user_message: str
    max_new_tokens: int = 2048
    temperature: float = 0.8123

# --- Define Lifespan for Model Loading/Unloading ---
@asynccontextmanager
async def lifespan(app: FastAPI):
    """
    Loads the quantized model into memory on startup and releases it on shutdown.
    """
    global model, tokenizer, max_seq_length
    print(f"Loading model from: {MODEL_PATH}...")

    if torch.cuda.is_available():
        print("Clearing CUDA cache...")
        torch.cuda.empty_cache()

    if os.path.isdir(MODEL_PATH):
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name = MODEL_PATH,
                max_seq_length = max_seq_length,
                dtype = None,
                load_in_4bit = True,
            )
            print("Model and Tokenizer loaded successfully.")
        except Exception as e:
            print(f"ERROR during model loading: {e}")
            model = None
            tokenizer = None
    else:
        print(f"ERROR: Model directory not found at {MODEL_PATH}")

    yield # <-- Application is ready to serve requests

    # --- SHUTDOWN LOGIC ---
    if 'model' in globals() and model is not None:
        del model
    if 'tokenizer' in globals() and tokenizer is not None:
        del tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Application shutting down. Model resources released.")


# --- FastAPI App Initialization ---
app = FastAPI(
    title="Mistral-7B Tourist Guide Chatbot API",
    version="1.0.0",
    lifespan=lifespan
)

# --- CORS Middleware Configuration ---
# This is ESSENTIAL for your local frontend to connect to the Colab API link.
# '*' allows all origins, which is fine for local development.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows ALL origins (your local frontend)
    allow_credentials=True,
    allow_methods=["*"],  # Allows POST, GET, etc.
    allow_headers=["*"],
)


# --- API Endpoint for Chat ---

@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    """
    Generates a tourist guide response based on the user's message.
    """
    if model is None or tokenizer is None:
        return {"error": "Model not loaded. Please check Colab logs."}

    # 1. Define the System Prompt for the Tourist Guide persona
    TOUR_GUIDE_PROMPT = (
        """
You are a seasoned and enthusiastic global tourist guide. Your primary goal is to provide insightful, detailed, and engaging guidance for various locations, cultures, and historical sites.

**Your Behavior Mandate (Strictly Follow All Rules):**
1.  **Persona:** Be knowledgeable, friendly, and prioritize providing practical, safety-conscious advice.
2.  **Reasoning & Detail:** For every recommendation or piece of information, you MUST provide a brief, logical reason (the "why") that supports your guidance.
3.  **Length Constraint:** All of your responses MUST be detailed but succinct. DO NOT exceed a length of 200 tokens (approximately 150-180 words) per answer.
4.  **Formatting:** Use **bolding** and **occasional travel-relevant emojis** for clarity. Organize famous sites, nearby points, and travel suggestions using **bullet points**.
5.  **Creative Storytelling & Scope:** Frame your answer like a **personal narrative or a thrilling story chapter** about the city. This narrative **MUST** include details of the city's famous places, highlight relevant nearby attraction points, and conclude with suggestions for other famous places the user can visit from the main city.
6.  **Suggestion & Sustainable Goals Related Ideas:** The answer must get end with texts like clean the environment or any other thing related to nature safety, do's and don'ts for nature safety.
7.  **Termination:** Your response **MUST** end with a concluding, encouraging remark to signal the end of the current guidance.
""")

    # 2. Format the Prompt using the model's chat template
    messages = [
        {"role": "system", "content": TOUR_GUIDE_PROMPT},
        {"role": "user", "content": request.user_message},
    ]

    prompt_formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 3. Tokenize and Generate
    inputs = tokenizer(
        prompt_formatted,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)

    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
            do_sample=True,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    except Exception as e:
        return {"error": f"Generation failed: {e}"}

    # 4. Decode the generated tokens
    decoded_output = tokenizer.decode(
        outputs[0, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    return {
        "response": decoded_output,
    }

In [ ]:
import asyncio
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import time # Import time for sleep

# The port where your Uvicorn server will run locally
LOCAL_PORT = 8000

# 0. Kill any process using the port if it's already in use
print(f"Attempting to clear port {LOCAL_PORT}...")
!fuser -k {LOCAL_PORT}/tcp || true # Use fuser to kill process on port, '|| true' to prevent error if no process
time.sleep(2) # Give the OS a moment to release the port
print(f"Port {LOCAL_PORT} cleared (if it was in use).")

# 1. Apply nest_asyncio (Still necessary to patch the loop)
nest_asyncio.apply()

# 2. Configure Uvicorn to run your 'app' on the local port
config = uvicorn.Config(
    app,                      # Your FastAPI instance
    host="0.0.0.0",
    port=LOCAL_PORT,
    loop="asyncio"
)

# 3. Create the Uvicorn Server instance
server = uvicorn.Server(config)

# 4. Define the asynchronous function to run everything
async def run_server_and_ngrok():
    print("Starting ngrok tunnel...")

    # Open the ngrok tunnel
    # This must happen before Uvicorn starts serving
    ngrok_tunnel = ngrok.connect(LOCAL_PORT)
    public_url = ngrok_tunnel.public_url

    print("\n\n#####################################################################")
    print(f"🎉 API Public URL (Backend Link): {public_url}")
    print(f"📖 Swagger Docs URL: {public_url}/docs")
    print("#####################################################################\n\n")

    # 5. Get the running event loop
    loop = asyncio.get_running_loop()

    # 6. Schedule the server's serve() coroutine as a task
    serve_task = loop.create_task(server.serve())

    # 7. Keep the loop running indefinitely while the server task is active
    try:
        await serve_task
    except asyncio.CancelledError:
        pass # Expected when the cell is stopped

# 8. Run the asynchronous function using the patched event loop
# This is the entry point that uses the patched loop correctly
asyncio.run(run_server_and_ngrok())

# 9. Clean up ngrok on exit (This part runs only after the asyncio.run() block exits)
ngrok.kill()
print("ngrok tunnel closed.")

Attempting to clear port 8000...
Port 8000 cleared (if it was in use).
Starting ngrok tunnel...


#####################################################################
🎉 API Public URL (Backend Link): https://jayceon-crumblier-unmeaningly.ngrok-free.dev
📖 Swagger Docs URL: https://jayceon-crumblier-unmeaningly.ngrok-free.dev/docs
#####################################################################




INFO:     Started server process [1680]
INFO:     Waiting for application startup.


Loading model from: /content/Mistral_7B_parameter_model_Quantasized...
Clearing CUDA cache...
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.0.0.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Model and Tokenizer loaded successfully.
INFO:     2409:40d2:1029:e2be:df8e:e449:b535:4531:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40d2:1029:e2be:df8e:e449:b535:4531:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2409:40d2:1029:e2be:df8e:e449:b535:4531:0 - "POST /chat HTTP/1.1" 200 OK
